# D1-0: sensor-stream opportunity gate on the Server Machine Dataset (validation machines only; the test machines are never downloaded)

Rung-0 study of the D1 domain. The question, the domain card, every frozen choice, the decision rules and the list of changes made before this run are in `docs/plans/d1-0-plan.md`; read that first. **This notebook is an exploratory opportunity gate, not a confirmatory test, and it makes no statement about H2**: it runs non-learned policies only, and nothing here is a co-state. It runs on CPU and needs network access to `raw.githubusercontent.com` for the data (MIT licence, register id `smd`; the data stay out of git).

**Design in one paragraph.** A window is 60 minutes of a 38-channel server-metrics stream. It starts with a snapshot of every channel; the allocator opens `k` channels (observed at every minute) and the rest are held at their snapshot value. The loss is the squared difference between the log-score of a frozen Mahalanobis detector on this partial stream and on the full stream. Machines are split by index mod 4 into tuning (7), validation (7) and test (14, never downloaded). The tuning machines choose the detector's `(floor, ridge)` and the best fixed allocation. The validation machines give the gate: does the best-known allocation (privileged greedy oracle or any deployable policy, per window) beat the best fixed allocation by at least 15 % of the normalised area, at equal sensing budget?

**Order of the notebook.** Tuning machines are downloaded and used; the correctness checks run on real windows; everything is frozen and hashed; only then are the validation machines downloaded.

In [ ]:
# ================================================================
# 0. Package (CPU is enough)
# ================================================================
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA
REPO_DIR = '/content/para_001' if IN_COLAB else str(Path.cwd().resolve().parents[1])

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)
    for candidate in (f'origin/{REPO_REF}', REPO_REF):
        resolved = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
                                  capture_output=True, text=True)
        if resolved.returncode == 0:
            subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolved.stdout.strip()], check=True)
            break
    else:
        raise RuntimeError(f'Cannot resolve {REPO_REF!r}.')
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
try:
    from adjointrwm.domains import sensor  # noqa: F401
except ImportError as error:
    raise RuntimeError(f'{REPO_REF!r} predates the D1 sensor module; set REPO_REF to a commit that has it.') from error

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    RUN_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs')
    DATA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/data/smd')
else:
    RUN_ROOT = Path.home() / 'adjointrwm_runs'
    DATA_ROOT = Path.home() / 'adjointrwm_data' / 'smd'
print('adjointrwm at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '', '| runs ->', RUN_ROOT, '| data ->', DATA_ROOT)

In [ ]:
# ================================================================
# 1. Frozen configuration, the immutable run directory and the tuning machines
# ================================================================
import datetime
import json
import platform
import tempfile
import time
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from adjointrwm.domains import sensor as sn
from adjointrwm.io import atomic_write_json, atomic_write_text, sha256_json


@dataclass(frozen=True)
class D10Config:
    primary_length: int = 60
    sensitivity_lengths: tuple = (30, 120)
    budgets: tuple = sn.DEFAULT_BUDGETS
    shrinkage: float = 0.1
    quantile: float = 0.995
    floors: tuple = (0.01, 0.05)
    ridges: tuple = (1e-6, 1e-2, 1e-1)
    min_relative_headroom: float = 0.15
    bootstrap_resamples: int = 10000
    bootstrap_bank: int = 20000
    bootstrap_seed: int = 0


CFG = D10Config()
CONFIG_DICT = asdict(CFG)
TUNING_MACHINES = sn.split_machines('tuning')
VALIDATION_MACHINES = sn.split_machines('validation')
SPLIT_TABLE = {split: list(sn.split_machines(split)) for split in sn.SPLITS}
CONFIG_HASH = sha256_json({'config': CONFIG_DICT, 'splits': SPLIT_TABLE})
RUN_ID = 'd1_0_sensor_opportunity_' + datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} exists; run directories are immutable.')
PATHS = {k: RUN_DIR / k for k in ('config', 'artifacts', 'figures', 'reports', 'tests')}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

TUNING_ROWS = [row for machine in TUNING_MACHINES for row in sn.download_machine(DATA_ROOT, machine)]
atomic_write_json(PATHS['config'] / 'data_manifest_tuning.json', {'files': TUNING_ROWS, 'sha256': sha256_json(TUNING_ROWS)})
TUNING = {machine: sn.load_machine(DATA_ROOT, machine) for machine in TUNING_MACHINES}

atomic_write_json(PATHS['config'] / 'run_config.json', {
    'run_id': RUN_ID, 'config': CONFIG_DICT, 'splits': SPLIT_TABLE, 'config_sha256': CONFIG_HASH,
    'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY, 'notebook': 'notebooks/04-domains/d1_0_sensor_opportunity.ipynb',
    'plan': 'docs/plans/d1-0-plan.md', 'domain_spec': sn.SPEC.to_dict(), 'domain_spec_sha256': sn.SPEC.checksum(),
    'test_machines_downloaded': False,
})
atomic_write_json(PATHS['config'] / 'environment.json', {
    'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'hardware': 'CPU (no accelerator needed)',
})
print('Run:', RUN_ID, '| config', CONFIG_HASH[:12], '|', len(TUNING), 'tuning machines,',
      sum(r['bytes'] for r in TUNING_ROWS) // 1_000_000, 'MB')

In [ ]:
# ================================================================
# 2. Tuning machines: the detector's (floor, ridge), then the best fixed family at each window length
# ================================================================
LENGTHS = (CFG.primary_length,) + CFG.sensitivity_lengths
SELECTION = sn.select_detector(list(TUNING.values()), CFG.primary_length, CFG.floors, CFG.ridges, CFG.shrinkage, CFG.quantile)
FLOOR, RIDGE = SELECTION['selected']['floor'], SELECTION['selected']['ridge']


def fit_all(machines, length):
    return {m: sn.fit_detector(d.train, length, FLOOR, RIDGE, CFG.shrinkage, CFG.quantile) for m, d in machines.items()}


BEST_FIXED, TUNING_AREAS = {}, {}
for length in LENGTHS:
    dets = fit_all(TUNING, length)
    results = {m: sn.evaluate_machine(dets[m], TUNING[m], length, CFG.budgets, 'score', with_oracle=False) for m in TUNING}
    chosen = sn.choose_best_fixed(results)
    BEST_FIXED[length], TUNING_AREAS[length] = chosen['selected'], chosen['areas']

atomic_write_json(PATHS['artifacts'] / 'detector_selection.json', SELECTION)
atomic_write_json(PATHS['artifacts'] / 'best_fixed_tuning.json', {'selected': {str(k): v for k, v in BEST_FIXED.items()},
                                                                  'areas': {str(k): v for k, v in TUNING_AREAS.items()}})
print(pd.DataFrame([{k: r[k] for k in ('floor', 'ridge', 'mean_f1')} | {'median_alarm_rate_normal': float(np.median(list(r['per_machine_alarm_rate_normal'].values())))}
                    for r in SELECTION['grid']]).to_string(index=False))
print('selected floor, ridge:', FLOOR, RIDGE, '| best fixed per length:', BEST_FIXED)
print(pd.DataFrame(TUNING_AREAS).round(4).to_string())

In [ ]:
# ================================================================
# 3. Correctness checks on real tuning windows (G3), before anything from validation exists
# ================================================================
CORRECTNESS = {}
machine0 = TUNING_MACHINES[0]
det0 = fit_all(TUNING, CFG.primary_length)[machine0]
wins0 = sn.make_windows(det0.standardise(TUNING[machine0].test), TUNING[machine0].labels, CFG.primary_length)
s_full0 = det0.score(wins0.z)
n_win, n_ch = len(wins0.index), wins0.z.shape[2]

parts = {s: set(sn.split_machines(s)) for s in sn.SPLITS}
CORRECTNESS['split_is_a_partition'] = bool(set.union(*parts.values()) == set(sn.machine_ids()) and sum(len(p) for p in parts.values()) == len(sn.machine_ids())
                                           and [len(parts[s]) for s in sn.SPLITS] == [7, 7, 14])
spy = []
try:
    with tempfile.TemporaryDirectory() as tmp:
        sn.download_machine(tmp, sn.split_machines('test')[0], fetch=lambda url: spy.append(url) or b'')
    CORRECTNESS['fetch_refuses_a_test_machine'] = False
except PermissionError:
    CORRECTNESS['fetch_refuses_a_test_machine'] = spy == []
eig = np.linalg.eigvalsh(det0.precision)
CORRECTNESS['precision_symmetric_positive_definite'] = bool(np.allclose(det0.precision, det0.precision.T) and eig.min() > 0)
CORRECTNESS['precision_condition_number'] = float(eig.max() / eig.min())
train_scores = det0.score(det0.standardise(TUNING[machine0].train))
CORRECTNESS['threshold_is_the_train_quantile'] = bool(np.isclose(det0.threshold, np.quantile(train_scores, CFG.quantile)))
CORRECTNESS['full_observation_has_zero_loss'] = bool(all(
    np.all(sn.objective_value(det0.score(sn.observed(wins0.z, np.ones((n_win, n_ch), bool))), s_full0, det0.threshold, kind) == 0) for kind in sn.OBJECTIVES))

worst = {}
for kind in sn.OBJECTIVES:
    gaps = []
    for direction in ('forward', 'backward'):
        j_by_size, order = sn.oracle_greedy(det0, wins0, kind, direction)
        for size in range(n_ch + 1):
            direct = sn.objective_value(det0.score(sn.observed(wins0.z, sn.oracle_mask(order, size, direction))), s_full0, det0.threshold, kind)
            gaps.append(float(np.abs(j_by_size[:, size] - direct).max()))
    worst[kind] = max(gaps)
CORRECTNESS['greedy_incremental_update_max_abs_difference'] = worst
CORRECTNESS['greedy_incremental_update_matches_direct'] = bool(max(worst.values()) < 1e-8)

perturbed_z = wins0.z.copy()
perturbed_z[:, 1:, :] = np.random.default_rng(0).normal(size=perturbed_z[:, 1:, :].shape) * 100
perturbed = sn.Windows(perturbed_z, wins0.prev1, wins0.prev2, wins0.labels, wins0.index, wins0.length)
CORRECTNESS['deployable_policies_ignore_the_window_after_the_snapshot'] = bool(all(
    np.array_equal(sn.policy_mask(name, k, det0, wins0), sn.policy_mask(name, k, det0, perturbed))
    for name in sn.DEPLOYABLE_POLICIES for k in (1, 4, 16)))
CORRECTNESS['machine_used'] = machine0
CORRECTNESS['passed'] = bool(all(v for k, v in CORRECTNESS.items() if isinstance(v, bool)))
atomic_write_json(PATHS['tests'] / 'correctness.json', CORRECTNESS)
print(json.dumps(CORRECTNESS, indent=2))
assert CORRECTNESS['passed'], 'G3 failed: do not read validation until this is fixed'

In [ ]:
# ================================================================
# 4. Freeze everything chosen on the tuning machines, then (and only then) fetch the validation machines
# ================================================================
FROZEN = {'config_sha256': CONFIG_HASH, 'tuning_manifest_sha256': sha256_json(TUNING_ROWS),
          'detector': {'floor': FLOOR, 'ridge': RIDGE, 'shrinkage': CFG.shrinkage, 'quantile': CFG.quantile},
          'best_fixed': {str(k): v for k, v in BEST_FIXED.items()}, 'primary_length': CFG.primary_length,
          'rules': {'G1': f'relative headroom >= {CFG.min_relative_headroom} at the primary length (point estimate); robust if the 95 % interval lower bound is too',
                    'G2': 'descriptive: retained fraction of the headroom and paired area differences', 'G3': 'correctness on real tuning windows'},
          'written_utc': datetime.datetime.now(datetime.UTC).isoformat()}
FROZEN['sha256'] = sha256_json({k: v for k, v in FROZEN.items() if k != 'written_utc'})
atomic_write_json(PATHS['reports'] / 'frozen_before_validation.json', FROZEN)
print('frozen', FROZEN['sha256'][:12], '| best fixed:', FROZEN['best_fixed'])

VALIDATION_ROWS = [row for machine in VALIDATION_MACHINES for row in sn.download_machine(DATA_ROOT, machine)]
atomic_write_json(PATHS['config'] / 'data_manifest_validation.json', {'files': VALIDATION_ROWS, 'sha256': sha256_json(VALIDATION_ROWS)})
VALIDATION = {machine: sn.load_machine(DATA_ROOT, machine) for machine in VALIDATION_MACHINES}
present_test = [m for m in sn.split_machines('test') if any(sn.machine_path(DATA_ROOT, kind, m).exists() for kind in sn.FILE_KINDS)]
assert not present_test, f'test machines found in the data root: {present_test}'
print(len(VALIDATION), 'validation machines,', sum(r['bytes'] for r in VALIDATION_ROWS) // 1_000_000, 'MB')

In [ ]:
# ================================================================
# 5. Validation machines: primary run, the decision-level objective, and the two other window lengths
# ================================================================
RUNS = {'primary': (CFG.primary_length, 'score')}
RUNS['decision'] = (CFG.primary_length, 'decision')
for length in CFG.sensitivity_lengths:
    RUNS[f'L{length}'] = (length, 'score')

RESULTS, FULL_F1, TIMINGS = {}, {}, {}
for run, (length, objective) in RUNS.items():
    started = time.time()
    dets = fit_all(VALIDATION, length)
    RESULTS[run] = {m: sn.evaluate_machine(dets[m], VALIDATION[m], length, CFG.budgets, objective) for m in VALIDATION}
    TIMINGS[run] = round(time.time() - started, 1)
    if run == 'primary':
        FULL_F1 = {m: sn.full_stream_f1(dets[m], VALIDATION[m].test, VALIDATION[m].labels) for m in VALIDATION}
        frames = []
        for m, r in RESULTS[run].items():
            window_index = sn.make_windows(dets[m].standardise(VALIDATION[m].test), VALIDATION[m].labels, length).index
            for name, table in r.tables.items():
                frames.append(pd.DataFrame({'machine': m, 'window': np.repeat(window_index, len(r.budgets)), 'policy': name,
                                            'budget': np.tile(np.array(r.budgets), len(window_index)), 'loss': table.ravel()}))
        pd.concat(frames).to_parquet(PATHS['artifacts'] / 'validation_window_losses_primary.parquet', index=False)
    print(run, 'L =', length, objective, '|', TIMINGS[run], 's |', {m: r.n_windows for m, r in RESULTS[run].items()})

In [ ]:
# ================================================================
# 6. Gate G1, descriptive G2, sensitivities, table, figure and report
# ================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SUMMARY = {}
for run, (length, objective) in RUNS.items():
    summary = sn.gate_summary(RESULTS[run], BEST_FIXED[length], min_relative=CFG.min_relative_headroom,
                              num_resamples=CFG.bootstrap_resamples, seed=CFG.bootstrap_seed, bank=CFG.bootstrap_bank)
    summary.update({'length': length, 'objective': objective, 'f1_by_budget': sn.pooled_f1(RESULTS[run]), 'budgets': list(CFG.budgets),
                    'machines': {m: {'windows': r.n_windows, 'anomalous_windows': r.n_anomalous_windows} for m, r in RESULTS[run].items()}})
    SUMMARY[run] = summary
atomic_write_json(PATHS['artifacts'] / 'validation_summary.json', {
    'runs': SUMMARY, 'full_observation_f1': FULL_F1, 'wall_clock_seconds': TIMINGS, 'validation_machines': list(VALIDATION_MACHINES)})

PHASE = pd.DataFrame([{'run': run, 'length': s['length'], 'objective': s['objective'], 'best_fixed': s['fixed'], 'area_fixed': s['areas'][s['fixed']],
                       'area_reference': s['areas'][sn.REFERENCE], 'area_oracle_greedy': s['areas'][sn.ORACLE],
                       'relative_headroom': s['relative_headroom'], 'ci_low': s['bootstrap']['headroom_ci'][0], 'ci_high': s['bootstrap']['headroom_ci'][1],
                       'passes_G1': s['passes'], 'robust': s['robust']} for run, s in SUMMARY.items()])
PHASE.to_csv(PATHS['figures'] / 'phase_table.csv', index=False)

primary = SUMMARY['primary']
fig, ax = plt.subplots(figsize=(7.5, 4.6))
curves = {m: sn.machine_curves(r) for m, r in RESULTS['primary'].items()}
for name in list(sn.DEPLOYABLE_POLICIES) + [sn.ORACLE, sn.REFERENCE]:
    mean_curve = np.mean([curves[m][name] for m in curves], axis=0)
    ax.plot(CFG.budgets, mean_curve, marker='o', ms=3, lw=2 if name in (BEST_FIXED[CFG.primary_length], sn.REFERENCE) else 1.1,
            ls='--' if name in (sn.ORACLE, sn.REFERENCE) else '-', label=name)
ax.set_xlabel('channels opened per window (k)')
ax.set_ylabel('loss / hold-only loss (mean over machines)')
ax.set_title(f'{RUN_ID}: validation machines, primary run', fontsize=9)
ax.legend(fontsize=7)
fig.savefig(PATHS['figures'] / 'loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

REPORT = {
    'run_id': RUN_ID, 'milestone': 'D1-0 (exploratory Rung-0 opportunity gate)', 'config_hash': CONFIG_HASH, 'frozen_sha256': FROZEN['sha256'], 'repo_commit': REPO_COMMIT,
    'correctness': CORRECTNESS, 'detector': FROZEN['detector'], 'best_fixed': FROZEN['best_fixed'],
    'G1_primary': {k: primary[k] for k in ('fixed', 'relative_headroom', 'passes', 'robust')} | {'ci': primary['bootstrap']['headroom_ci']},
    'G1_by_run': {run: {k: s[k] for k in ('fixed', 'relative_headroom', 'passes', 'robust')} | {'ci': s['bootstrap']['headroom_ci']} for run, s in SUMMARY.items()},
    'test_machines_downloaded': False, 'validation_machines': list(VALIDATION_MACHINES),
    'claim_boundary': ('Exploratory Rung-0 gate on seven validation machines of the Server Machine Dataset with non-learned policies. The oracle is a privileged greedy '
                       'selection and the reference is the best-known curve over it and every deployable policy. The loss is a score-fidelity proxy for a frozen detector. '
                       'No learned critic and no co-state are involved, so no H2 statement follows. Policy computation is not charged.'),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
lines = [f'# D1-0 `{RUN_ID}` (exploratory)', '', f"Correctness: **{'PASS' if CORRECTNESS['passed'] else 'FAIL'}** · G1 at the primary length: "
         f"**{'PASS' if primary['passes'] else 'FAIL'}**{' (robust)' if primary['robust'] else ' (not robust)'} · relative headroom {primary['relative_headroom']:.3f} "
         f"[{primary['bootstrap']['headroom_ci'][0]:.3f}, {primary['bootstrap']['headroom_ci'][1]:.3f}]", '',
         '| Run | Length | Objective | Best fixed | Relative headroom | 95 % interval | G1 | Robust |', '|---|---:|---|---|---:|---|---|---|']
for _, r in PHASE.iterrows():
    lines.append(f"| {r['run']} | {r['length']} | {r['objective']} | {r['best_fixed']} | {r['relative_headroom']:.3f} | [{r['ci_low']:.3f}, {r['ci_high']:.3f}] | {r['passes_G1']} | {r['robust']} |")
lines += ['', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)

## Reading the result

- `reports/run_summary.md` and `reports/acceptance_report.json` give G1 for each run. **The primary run is the one the gate is decided on**; the other runs are the pre-registered sensitivities (decision-level objective, `L` = 30 and 120).
- G1 passes only if the relative headroom is at least 0.15; it is **robust** only if the lower end of the 95 % interval is too. Seven machines make the interval wide by construction.
- `artifacts/validation_summary.json` holds the areas, the fraction of the headroom that each dynamic policy keeps (G2), paired area differences, per-machine headroom and detection F1 by budget.
- A pass licenses a Rung-1 design and needs Roman's go-ahead; a fail means redesigning the candidates (plan §6). Neither is a statement about H2.